# Milestone 1: Formula 1 Podium Prediction Companion
**Course:** CSEN 903 · Advanced Machine Learning & Data Engineering  
**Project:** F1 World Championship (1950–2024) Podium Prediction Pipeline  
**Target:** Binary classification — whether a driver finishes on the podium (`positionOrder <= 3`).  
**Prediction Temporal Cutoff:** Strictly *after qualifying, before race start* (lights out). Current-race outcome fields and post-race standings are strictly forbidden as features.

---

### Team Ownership Matrix

| Member | Role | Pipeline Ownership |
| :--- | :--- | :--- |
| **M1** | Data Lead | Raw table audit, referential integrity, cleaning, duplicate resolution, before/after EDA, report assembly |
| **M2** | Analytics Lead | Data-engineering multi-hop questions (Q1–Q3), domain mappings, Explainable AI (XAI) suite |
| **M3** | Features Lead | Leakage-safe feature engineering, temporal validation, preprocessing pipelines, order experiments, ablation |
| **M4** | Modeling Lead | Heuristic & statistical baselines, Shallow Neural Network, hyperparameter tuning, threshold calibration, inference function |

---

### Pipeline Roadmap
1. **Shared Components & Global Configuration** *(Current Section)*
2. **Data Audit & Integrity Verification** *(Step 1 · M1)*
3. **Data Cleaning & Grain Resolution** *(Step 2 · M1)*
4. **Before vs. After Exploratory Data Analysis (EDA)** *(Step 2 · M1)*
5. **Data-Engineering Analytical Questions (Q1–Q3)** *(Step 3 · M2)*
6. **Leakage-Safe Feature Engineering** *(Step 4 · M3)*
7. **Preprocessing Pipelines & Splitting Experiments** *(Step 5 · M3)*
8. **Predictive Modeling Benchmarks & Tuning** *(Step 6 · M4)*
9. **Feature Group Ablation Study** *(Step 7 · M3)*
10. **Explainable AI (XAI: SHAP & LIME)** *(Step 8 · M2)*
11. **Operational Inference Interface** *(Step 9 · M4)*
12. **Limitations & Project Summary** *(Step 10 · All)*

## 1. Shared Components (Day 1 Agreement)

To maintain strict pipeline integrity across all team members and prevent data leakage or environment mismatches, all core configurations, data contracts, and visual standards are formalized below:

1. **Config Cell**: Single source of truth for random seeds, split cutoffs, file paths, and historical race scope flags.
2. **Leakage Blacklist**: An enforced registry of current-race outcome fields across all 14 tables that must never enter feature engineering.
3. **Modeling Data Contract**: Strict specification of the modeling grain `(raceId, driverId)`, target column, identifier lineage, and `FEATURE_GROUPS` dictionary.
4. **Unified Visual Style & Plot Exporter**: Publication-quality Seaborn/Matplotlib styling exporting 300-DPI figures directly to `figures/`.
5. **Decision Log**: Traceable registry of all architectural, scope, and algorithmic choices.

In [ ]:
# ==============================================================================
# 1.1 SHARED CONFIGURATION & ENVIRONMENT SETUP
# ==============================================================================
from dataclasses import dataclass
from pathlib import Path
import os
import sys
import re
import warnings
from typing import Dict, List, Set, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress minor warnings for clean presentation
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Set global random seed for complete reproducibility
RANDOM_SEED: int = 42
np.random.seed(RANDOM_SEED)

# Temporal Split Boundaries (Temporal splitting prevents regulation leakage)
# Train: <= 2019 | Validation: 2020-2021 | Test: >= 2022 (Modern ground-effect era)
TRAIN_MAX_YEAR: int = 2019
VAL_YEARS: Tuple[int, int] = (2020, 2021)
TEST_MIN_YEAR: int = 2022

# Scope & Cleaning Flags
EXCLUDE_INDY_500: bool = True  # Exclude 1950-1960 Indy 500 rounds (non-F1 formula)
SENTINEL_VALUES: List[str] = [r"\N", "\\N", ""]

# Output directories
FIGURES_DIR: Path = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Dynamic Dataset Path Resolution (Kaggle & Local fallback)
def resolve_dataset_dir() -> Path:
    candidates = [
        Path("/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020"),
        Path("/kaggle/input/formula-1-world-championship-1950-2020"),
        Path("./archive"),
        Path("../archive"),
        Path("d:/GUC/Semester 9/ACL/Project/Formula 1/archive"),
    ]
    for p in candidates:
        if p.exists() and (p / "results.csv").exists():
            return p.resolve()
    # Dynamic Kaggle fallback: recursively locate results.csv under /kaggle/input
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for sub in kaggle_input.rglob("results.csv"):
            return sub.parent.resolve()
    raise FileNotFoundError(
        f"F1 dataset directory could not be located. Checked paths:\n"
        + "\n".join(f" - {p}" for p in candidates)
    )

DATASET_DIR: Path = resolve_dataset_dir()

# Set Pandas presentation options
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 1000)

print(f"[OK] Random Seed        : {RANDOM_SEED}")
print(f"[OK] Dataset Directory  : {DATASET_DIR}")
print(f"[OK] Figures Directory  : {FIGURES_DIR.resolve()}")
print(f"[OK] Split Years        : Train <= {TRAIN_MAX_YEAR} | Val {VAL_YEARS} | Test >= {TEST_MIN_YEAR}")
print(f"[OK] Exclude Indy 500   : {EXCLUDE_INDY_500}")

In [ ]:
# ==============================================================================
# 1.2 UNIFIED VISUALIZATION STYLING & FIGURE EXPORT HELPER
# ==============================================================================

# Custom F1-inspired, publication-grade visual theme
F1_PALETTE = {
    "crimson": "#E10600",      # Formula 1 Official Red
    "dark": "#15151E",         # Charcoal Black
    "teal": "#00D2BE",         # Mercedes / Modern Cyan
    "blue": "#1E41FF",         # Red Bull / Alpine Blue
    "orange": "#FF8700",       # McLaren Papaya
    "green": "#229971",        # Aston Martin Green
    "gray": "#7A7A85",         # Neutral Muted Gray
    "light_gray": "#E0E0E6",   # Background Grid Line Gray
}

# Apply global Matplotlib & Seaborn styling
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 100,
    "figure.autolayout": False,
    "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Helvetica", "Arial"],
    "font.family": "sans-serif",
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.titlepad": 14,
    "axes.labelsize": 12,
    "axes.labelweight": "medium",
    "axes.labelpad": 8,
    "axes.edgecolor": "#CCCCCC",
    "axes.linewidth": 1.0,
    "grid.color": "#E5E5EB",
    "grid.linestyle": "--",
    "grid.alpha": 0.7,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "legend.frameon": True,
    "legend.framealpha": 0.9,
    "legend.edgecolor": "#E5E5EB",
})

def save_fig(
    fig_id: str,
    tight_layout: bool = True,
    fig_extension: str = "png",
    dpi: int = 300,
    transparent: bool = False
) -> Path:
    """
    Saves the active matplotlib figure to the shared FIGURES_DIR with consistent formatting.
    
    Parameters
    ----------
    fig_id : str
        Unique snake_case identifier for the figure (e.g., 'q1_front_row_conversion').
    tight_layout : bool, default=True
        Whether to invoke plt.tight_layout() prior to export.
    fig_extension : str, default='png'
        File extension format ('png', 'pdf', 'svg').
    dpi : int, default=300
        Resolution for publication-quality output.
    transparent : bool, default=False
        Whether background should be saved as transparent.
        
    Returns
    -------
    Path : The saved file path.
    """
    file_path = FIGURES_DIR / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(file_path, format=fig_extension, dpi=dpi, transparent=transparent, bbox_inches="tight")
    print(f"[FIGURE SAVED] {file_path}")
    return file_path

print("[OK] Publication plot theme and save_fig() helper initialized.")

In [ ]:
# ==============================================================================
# 1.3 DATA LEAKAGE BLACKLIST & INTEGRITY GUARD (AUDITED ACROSS ALL 14 TABLES)
# ==============================================================================

# Core Blacklist: Columns strictly generated during or after the race.
# These can NEVER be used as features for predicting pre-race podium probability.
LEAKAGE_BLACKLIST_COLUMNS: Set[str] = {
    # 1. Current-race direct outcomes from results.csv
    "position",         # Race finishing position (e.g. '1', '2', '\N')
    "positionText",     # Finishing status text ('R', 'D', 'W', etc.)
    "positionOrder",    # Finishing order integer -> TARGET ONLY (podium = positionOrder <= 3)
    "points",           # Current race points scored (raw column name)
    "laps",             # Completed race laps (raw column name)
    "time",             # Race finish time / gap string (raw column name)
    "milliseconds",     # Total race time in milliseconds (raw column name)
    "fastestLap",       # Lap number on which fastest lap was set
    "rank",             # Fastest lap rank among drivers in the race
    "fastestLapTime",   # Lap time string of fastest lap in race
    "fastestLapSpeed",  # Average speed of fastest lap in race
    "statusId",         # Current race finishing/retirement status ID
    
    # 2. Joined outcome descriptions from status.csv & constructor_results.csv
    "status",           # Outcome description ('Finished', 'Engine', 'Collision', etc.)
    
    # 3. In-race telemetry from lap_times.csv & pit_stops.csv
    "lap",              # Lap number during race
    "stop",             # Pit stop number
    "duration",         # Pit stop duration
    
    # 4. Standings collisions (unlagged post-race championship standings)
    "wins",             # Post-race cumulative wins
    "post_race_points",
    "post_race_position",
    "post_race_wins",
    "post_race_rank",
}

# Regex patterns for catching accidental post-race or in-race column variants
LEAKAGE_PATTERNS: List[re.Pattern] = [
    re.compile(r"^fastest.*", re.IGNORECASE),
    re.compile(r"^post_.*", re.IGNORECASE),
    re.compile(r".*current.*", re.IGNORECASE),
    re.compile(r"^finish.*", re.IGNORECASE),
    re.compile(r"^pit_stop_.*", re.IGNORECASE),
    re.compile(r"^lap_time_.*", re.IGNORECASE),
]

# Table-by-table schema audit reference mapping:
TABLE_TEMPORAL_AUDIT: Dict[str, str] = {
    "circuits": "PRE-RACE SAFE: Static track characteristics (country, location, altitude, lat, lng).",
    "constructors": "PRE-RACE SAFE: Static constructor identity (name, nationality).",
    "drivers": "PRE-RACE SAFE: Static driver identity (dob/age, nationality).",
    "seasons": "PRE-RACE SAFE: Season year metadata.",
    "races": "PRE-RACE SAFE: Scheduled race calendar, round, circuit, session timestamps.",
    "qualifying": "PRE-RACE SAFE: Concluded before lights-out. Column 'position' MUST be renamed to 'quali_position'.",
    "results": "HYBRID: 'grid', 'constructorId', 'number' are SAFE; 'positionOrder' is TARGET; 11 outcome columns are BLACKLISTED.",
    "status": "POST-RACE LEAKAGE: Describes current race termination reason. Safe only for historical aggregations.",
    "constructor_results": "POST-RACE LEAKAGE: Records current race constructor points. Safe only lagged to t-1.",
    "constructor_standings": "POST-RACE LEAKAGE: Ergast records post-race standings for raceId. Safe only lagged to round t-1.",
    "driver_standings": "POST-RACE LEAKAGE: Ergast records post-race standings for raceId. Safe only lagged to round t-1.",
    "lap_times": "IN-RACE TELEMETRY: 100% Forbidden for current race.",
    "pit_stops": "IN-RACE EVENTS: 100% Forbidden for current race.",
    "sprint_results": "PRE-RACE SAFE (CONDITIONAL): Saturday sprint before Sunday race; features must be prefixed with 'sprint_'.",
}

def assert_leakage_safe(feature_cols: List[str]) -> None:
    """
    Validates candidate feature columns against the comprehensive leakage blacklist.
    Raises ValueError immediately if any blacklisted column or pattern is detected.
    """
    violations = []
    for col in feature_cols:
        col_clean = col.strip()
        if col_clean in LEAKAGE_BLACKLIST_COLUMNS:
            violations.append(f"Direct Blacklist Match: '{col_clean}'")
        for pattern in LEAKAGE_PATTERNS:
            if pattern.match(col_clean):
                violations.append(f"Pattern Match [{pattern.pattern}]: '{col_clean}'")
                
    if violations:
        error_msg = "\n".join(f"  ❌ {v}" for v in violations)
        raise ValueError(
            f"[CRITICAL DATA LEAKAGE DETECTED]\n"
            f"The following feature columns violate pre-race temporal cutoff:\n{error_msg}"
        )
    print(f"[LEAKAGE CHECK PASSED] All {len(feature_cols)} candidate features strictly respect pre-race boundary.")

# Sanity tests: Ensure validator catches direct, pattern, and standings collision leaks
test_failures = []
for forbidden in ["milliseconds", "position", "status", "fastestLapTime", "wins"]:
    try:
        assert_leakage_safe(["grid", "quali_position", forbidden])
        test_failures.append(forbidden)
    except ValueError:
        pass

if not test_failures:
    print("[TEST CONFIRMED] Leakage guard successfully verified against test forbidden columns.")
else:
    raise AssertionError(f"Leakage guard failed to catch: {test_failures}")

In [ ]:
# ==============================================================================
# 1.4 MODELING DATA CONTRACT & FEATURE GROUPS REGISTRY
# ==============================================================================

# 1. Grain & Lineage Identifiers (retained for indexing, never used as numeric features)
IDENTIFIER_COLUMNS: List[str] = [
    "raceId",
    "driverId",
    "constructorId",
    "date",
    "year",
]

# 2. Prediction Target
TARGET_COLUMN: str = "podium"  # Binary: 1 if positionOrder <= 3 else 0

# 3. Canonical Feature Groups Mapping
# Used for feature engineering, ablation studies, and XAI grouping.
FEATURE_GROUPS: Dict[str, List[str]] = {
    "grid": [
        "grid",
        "pit_start",
        "front_row",
        "teammate_grid_diff",
    ],
    "qualifying": [
        "quali_position",
        "gap_to_pole",
        "q3_reached",
        "missing_quali_flag",
    ],
    "driver_form": [
        "driver_rolling_podium_rate_5",
        "driver_rolling_avg_finish_5",
        "driver_career_podium_rate",
    ],
    "constructor_form": [
        "constructor_rolling_podium_rate_5",
        "constructor_rolling_points_5",
    ],
    "standings": [
        "driver_pre_race_points",
        "driver_pre_race_rank",
        "constructor_pre_race_points",
        "constructor_pre_race_rank",
    ],
    "context": [
        "field_size",
        "round",
        "is_home_race",
        "circuit_historical_podium_rate",
    ],
    "sprint": [
        "sprint_position",
        "sprint_points",
    ],
}

def validate_data_contract(
    df: pd.DataFrame,
    feature_cols: Optional[List[str]] = None,
    enforce_grain_only: bool = False
    ) -> bool:
    """
    Enforces the modeling table data contract:
    1. Composite grain uniqueness on (raceId, driverId).
    2. Presence and non-nullity of identifier columns.
    3. Target column presence, non-nullity, and binary values {0, 1}.
    4. Features pass the leakage guard.
    """
    # Check grain
    if not {"raceId", "driverId"}.issubset(df.columns):
        raise AssertionError("Dataframe is missing composite key columns 'raceId' or 'driverId'.")
    
    dup_count = df.duplicated(subset=["raceId", "driverId"]).sum()
    if dup_count > 0:
        raise AssertionError(
            f"Grain violation: Found {dup_count} duplicate (raceId, driverId) rows! "
            f"Analytical table must have strictly one row per driver per race."
        )
        
    if enforce_grain_only:
        print(f"[CONTRACT PASSED] Grain strictly verified: {len(df):,} unique (raceId, driverId) entries.")
        return True
        
    # Check identifiers
    for col in IDENTIFIER_COLUMNS:
        if col not in df.columns:
            raise AssertionError(f"Missing required identifier column: '{col}'")
        if df[col].isnull().any():
            raise AssertionError(f"Identifier column '{col}' contains unexpected NULL values.")
            
    # Check target
    if TARGET_COLUMN not in df.columns:
        raise AssertionError(f"Target column '{TARGET_COLUMN}' is missing from modeling table.")
    if df[TARGET_COLUMN].isnull().any():
        raise AssertionError(f"Target column '{TARGET_COLUMN}' contains NULL values.")
    unique_targets = set(df[TARGET_COLUMN].dropna().unique())
    if not unique_targets.issubset({0, 1}):
        raise AssertionError(f"Target '{TARGET_COLUMN}' must be strictly binary {{0, 1}}, found: {unique_targets}")
        
    # Validate feature columns if provided
    if feature_cols is not None:
        assert_leakage_safe(feature_cols)
        
    print(f"[CONTRACT PASSED] Full modeling table contract satisfied across {len(df):,} rows.")
    return True

print("[OK] Data contract rules and FEATURE_GROUPS registry established.")

### 1.5 Shared Decision Log & 14-Table Leakage Reference

The table below embeds the shared architectural and pipeline decisions established for Milestone 1. The persistent version is tracked in [`docs/decision_log.md`](docs/decision_log.md).

| Decision ID | Pipeline Stage | Owner | Decision | Rationale & Evidence | Status |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **DEC-001** | Shared Setup | All | Global Random Seed = `42` | Ensures deterministic reproduction across all environments and runs. | **Approved** |
| **DEC-002** | Shared Setup | M3 / All | Temporal Split: Train $\le 2019$, Val $2020\text{–}2021$, Test $\ge 2022$ | Prevents temporal leakage across F1 aero/engine regulation eras; 2022 tests ground-effect generalization. | **Approved** |
| **DEC-003** | Scope / Cleaning | M1 | Exclude Indianapolis 500 ($1950\text{–}1960$) | Run under AAA/USAC Indy regulations on an oval; non-F1 grid contaminates conversion & form baselines. | **Approved** |
| **DEC-004** | Leakage Guard | M3 / M4 | Strict pre-race temporal cutoff & Blacklist | Prediction target is `positionOrder <= 3`. All current-race in-event metrics across 14 tables are forbidden. | **Approved** |
| **DEC-005** | Data Contract | All | Table Grain strictly `(raceId, driverId)` | Exactly one prediction per driver-race entry. Identifiers retained for lineage, target is binary `podium`. | **Approved** |
| **DEC-006** | Environment | M1 | Kaggle & Local path auto-detection | Seamless execution across Kaggle notebooks and local git repositories with zero config tweaks. | **Approved** |
| **DEC-007** | Visual Standards | M2 | Publication-grade Seaborn style & `save_fig()` | Enforces visual cohesion and automatic 300-DPI exports to `figures/` for report assembly. | **Approved** |

#### 14-Table Leakage Audit Summary
- **Safe Pre-Race Tables:** `circuits`, `constructors`, `drivers`, `seasons`, `races`, `qualifying` (must rename `position` to `quali_position`).
- **Target & In-Race Table (`results`):** Only `grid`, `constructorId`, `number` are pre-race features; `positionOrder` defines target; remaining 11 columns are strictly blacklisted.
- **In-Race Telemetry Tables (100% Forbidden):** `lap_times`, `pit_stops`.
- **Post-Race Ergast Standings Tables:** `driver_standings`, `constructor_standings`, `constructor_results` are updated *after* `raceId`. Must be lagged to round $t-1$.

In [ ]:
# ==============================================================================
# 1.6 RAW DATASET LOADER (CLEAN COPY PATTERN)
# ==============================================================================

def load_raw_datasets(dataset_dir: Path, sentinels: List[str] = SENTINEL_VALUES) -> Dict[str, pd.DataFrame]:
    """
    Loads all 14 relational CSV tables with standardized sentinel values into memory.
    
    CRITICAL RULE:
    The returned dataframes in raw_dataframes must remain 100% IMMUTABLE.
    All downstream auditing, cleaning, and transformations must operate strictly 
    on explicit copies (.copy()) to preserve original raw state for before/after EDA.
    """
    csv_paths = sorted(dataset_dir.glob("*.csv"))
    if not csv_paths:
        raise FileNotFoundError(f"No CSV files found in {dataset_dir}")
        
    tables: Dict[str, pd.DataFrame] = {}
    summary_rows = []
    
    for file_path in csv_paths:
        table_name = file_path.stem
        # Read with sentinel handling
        df = pd.read_csv(file_path, na_values=sentinels, low_memory=False)
        tables[table_name] = df
        
        summary_rows.append({
            "Table Name": table_name,
            "File Name": file_path.name,
            "Rows": f"{len(df):,}",
            "Columns": len(df.columns),
            "Memory (MB)": f"{df.memory_usage(deep=True).sum() / (1024 * 1024):.2f}",
        })
        
    summary_df = pd.DataFrame(summary_rows)
    print(f"[SUCCESS] Loaded {len(tables)} relational F1 tables into memory.")
    display(summary_df)
    return tables

# Execute the raw loader
raw_dataframes = load_raw_datasets(DATASET_DIR)

---

# 2. Step 1: Data Audit & Structural Inspection (M1 · Data Lead)

> **Owner:** Data Lead (**M1**)  
> **Prerequisite:** Shared Components initialized (`raw_dataframes` loaded, leakage guards active, `save_fig()` configured).  
> **Scope:** Establish the complete structural baseline across all 14 relational tables, audit schema constraints, quantify memory and missingness footprint, and profile key pipeline tables prior to cleaning.

---

## 2.1 Task 1: Raw Table Inventory & Structural Inspection

**Core Objectives:**
1. **Inventory Assertions:** Programmatically assert that `raw_dataframes` contains exactly 14 keys matching canonical Ergast schema names.
2. **Comprehensive Structural Summary Matrix:** Tabulate physical row counts, column counts, memory footprints, missing cell volumes, null rates, and data type breakdowns across all tables.
3. **Per-Table Structural Profiling:** Execute a standardized inspection loop generating `.info()` schema layouts and `.describe(include='all')` statistical distributions.
4. **Focal Audit on Key Tables:**
   - **`results`** (26,759 rows × 18 cols): The pipeline spine housing the prediction target (`positionOrder <= 3`), 11 blacklisted post-race columns, and 176 shared-drive composite key duplicates.
   - **`lap_times`** (589,081 rows × 6 cols): The heavyweight telemetry table (85.3% of total records), confirming 100% pre-race exclusion under [DEC-004](docs/decision_log.md).
   - **`qualifying`** (10,494 rows × 9 cols): The starting performance table; quantifying historical coverage truncation (1994+) and expected knockout missingness (`q2`/`q3`).

> [!IMPORTANT]
> **Clean Working Copy Rule:** `raw_dataframes` remains strictly immutable. All structural audits operate in read-only mode to preserve the raw baseline for before/after comparisons.

In [ ]:
# ==============================================================================
# 2.1.1 TASK 1.1: PROGRAMMATIC TABLE INVENTORY & INTEGRITY ASSERTIONS
# ==============================================================================

# Canonical registry of the 14 expected Ergast relational tables
EXPECTED_TABLES: Set[str] = {
    "circuits",
    "constructor_results",
    "constructor_standings",
    "constructors",
    "driver_standings",
    "drivers",
    "lap_times",
    "pit_stops",
    "qualifying",
    "races",
    "results",
    "seasons",
    "sprint_results",
    "status",
}

# 1. Assert exact count and naming of tables in raw_dataframes
loaded_keys = set(raw_dataframes.keys())
missing_tables = EXPECTED_TABLES - loaded_keys
unexpected_tables = loaded_keys - EXPECTED_TABLES

assert len(raw_dataframes) == 14, (
    f"[ERROR] Expected exactly 14 relational tables, but found {len(raw_dataframes)}: {loaded_keys}"
)
assert not missing_tables, f"[ERROR] Missing expected tables: {missing_tables}"
assert not unexpected_tables, f"[ERROR] Unexpected extra tables: {unexpected_tables}"

# 2. Assert all loaded objects are non-empty DataFrames with valid shapes
empty_tables = []
for name, df in raw_dataframes.items():
    if not isinstance(df, pd.DataFrame) or df.empty:
        empty_tables.append(name)

assert not empty_tables, f"[ERROR] Tables loaded as empty or non-DataFrame: {empty_tables}"

print("[SUCCESS] All 14 canonical tables verified in raw_dataframes:")
for idx, tbl in enumerate(sorted(raw_dataframes.keys()), 1):
    df_tbl = raw_dataframes[tbl]
    print(f"  {idx:2d}. {tbl:<23} | Shape: {df_tbl.shape[0]:>7,} rows × {df_tbl.shape[1]:>2} cols")

In [ ]:
# ==============================================================================
# 2.1.2 TASK 1.2: COMPREHENSIVE 14-TABLE STRUCTURAL SUMMARY TABLE
# ==============================================================================

# Candidate natural grain and temporal safety mappings per cleaning_notes.md
TABLE_METADATA: Dict[str, Dict[str, str]] = {
    "circuits": {"grain": "circuitId", "safety": "Pre-Race Safe (Static track characteristics)"},
    "constructors": {"grain": "constructorId", "safety": "Pre-Race Safe (Static team identity)"},
    "drivers": {"grain": "driverId", "safety": "Pre-Race Safe (Static driver demographics)"},
    "seasons": {"grain": "year", "safety": "Pre-Race Safe (Season calendar metadata)"},
    "races": {"grain": "raceId", "safety": "Pre-Race Safe (Race calendar & session timestamps)"},
    "qualifying": {"grain": "qualifyId / (raceId, driverId)", "safety": "Pre-Race Safe (Requires position rename)"},
    "results": {"grain": "resultId / (raceId, driverId)*", "safety": "Hybrid (Target + 11 blacklisted cols)"},
    "sprint_results": {"grain": "resultId / (raceId, driverId)", "safety": "Conditional Pre-Race (Requires sprint_ prefix)"},
    "status": {"grain": "statusId", "safety": "Post-Race Leakage (Current race outcome)"},
    "constructor_results": {"grain": "constructorResultsId", "safety": "Post-Race Leakage (Current race points)"},
    "constructor_standings": {"grain": "constructorStandingsId", "safety": "Ergast Trap (Post-race cumulative points)"},
    "driver_standings": {"grain": "driverStandingsId", "safety": "Ergast Trap (Post-race cumulative points)"},
    "lap_times": {"grain": "(raceId, driverId, lap)", "safety": "In-Race Telemetry (100% Forbidden)"},
    "pit_stops": {"grain": "(raceId, driverId, stop)", "safety": "In-Race Events (100% Forbidden)"},
}

summary_records = []
total_rows = 0
total_nulls = 0
total_cells = 0
total_memory_bytes = 0

for table_name in sorted(raw_dataframes.keys()):
    df = raw_dataframes[table_name]
    n_rows, n_cols = df.shape
    mem_bytes = df.memory_usage(deep=True).sum()
    null_cells = int(df.isna().sum().sum())
    cell_count = n_rows * n_cols
    null_pct = (null_cells / cell_count * 100) if cell_count > 0 else 0.0
    
    # Dtype breakdown
    int_cols = len(df.select_dtypes(include=["int64", "int32", "int16", "int8"]).columns)
    float_cols = len(df.select_dtypes(include=["float64", "float32"]).columns)
    obj_cols = len(df.select_dtypes(include=["object"]).columns)
    
    meta = TABLE_METADATA.get(table_name, {"grain": "Unknown", "safety": "Unclassified"})
    
    total_rows += n_rows
    total_nulls += null_cells
    total_cells += cell_count
    total_memory_bytes += mem_bytes
    
    summary_records.append({
        "Table": table_name,
        "Rows": n_rows,
        "Cols": n_cols,
        "Memory (MB)": round(mem_bytes / (1024 * 1024), 2),
        "Null Cells": null_cells,
        "Null %": round(null_pct, 2),
        "Int Cols": int_cols,
        "Float Cols": float_cols,
        "Object Cols": obj_cols,
        "Candidate Grain": meta["grain"],
        "Temporal Safety": meta["safety"],
    })

inventory_summary_df = pd.DataFrame(summary_records)

# Print database-wide aggregate totals
print("=" * 95)
print("F1 RELATIONAL DATABASE AGGREGATE SUMMARY (14 TABLES)")
print("=" * 95)
print(f"Total Relational Records : {total_rows:,} rows")
print(f"Total Columns Sum       : {inventory_summary_df['Cols'].sum()} columns across all tables")
print(f"Total Memory Footprint  : {total_memory_bytes / (1024 * 1024):.2f} MB")
print(f"Total Missing Cells     : {total_nulls:,} / {total_cells:,} ({total_nulls / total_cells * 100:.2f}% missingness overall)")
print("=" * 95)

# Display formatted summary table
display(inventory_summary_df)

### 2.1.3 Structural Audit Observations & Insights

An analysis of the structural summary reveals several foundational characteristics of the Ergast relational schema:

1. **Volume Imbalance & In-Race Telemetry:**
   - `lap_times` comprises **589,081 rows** (~27 MB), representing **85.3%** of all records in the database.
   - **Strict Leakage Guard:** Despite being the largest table, `lap_times` (along with `pit_stops` at 11,371 rows) records in-race events that occur *strictly after lights out*. Under [DEC-004](docs/decision_log.md), these tables are **100% blacklisted** for pre-race prediction.

2. **The Pipeline Spine (`results.csv`):**
   - Contains **26,759 rows × 18 columns**, representing every official driver entry across 1,125 World Championship Grands Prix (1950–2024).
   - Houses our binary prediction target (`podium = positionOrder <= 3`).
   - Contains 11 outcome columns that must be strictly quarantined from the feature space.
   - Contains **176 duplicate `(raceId, driverId)` rows** due to historical shared drives (1950s), audited in Task 2 and resolved in Task 6.

3. **Coverage Drift (`qualifying.csv`):**
   - Contains **10,494 rows × 9 columns**.
   - Ergast qualifying session data only starts in **1994**. Races from 1950 to 1993 have starting positions recorded only via `results.grid`.
   - Knockout qualifying (Q1/Q2/Q3) began in **2006**; missing values in `q2` and `q3` reflect driver eliminations (structural non-participation), not corrupt data.

4. **Sentinel Handling & Dtype Hygiene:**
   - Loading with `SENTINEL_VALUES = [r'\N', '\\N', '']` converted literal string sentinels to `np.nan`.
   - Several numeric columns (`results.milliseconds`, `results.fastestLap`, `drivers.number`, `pit_stops.duration`) loaded as `float64` or `object` due to nulls or mixed historical encodings. These will undergo explicit semantic typing in Task 4.

In [ ]:
# ==============================================================================
# 2.1.4 TASK 1.3: PER-TABLE STRUCTURAL PROFILING (.info() & .describe() SUITE)
# ==============================================================================
import io

def profile_f1_table(table_name: str, df: pd.DataFrame, show_sample: bool = True) -> None:
    """
    Renders a complete structural profile for an individual F1 table:
    1. Schema summary and memory footprint.
    2. Column-by-column .info() breakdown (non-null counts, dtypes).
    3. Comprehensive .describe(include='all') statistical distribution.
    4. Head preview showing raw sample rows.
    """
    mem_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
    print("=" * 95)
    print(f"TABLE PROFILE: {table_name.upper()} | {len(df):,} Rows × {len(df.columns)} Cols | Memory: {mem_mb:.2f} MB")
    print("=" * 95)
    
    # Capture .info() output into a clean string buffer
    buf = io.StringIO()
    df.info(buf=buf, verbose=True, show_counts=True)
    print(buf.getvalue())
    
    print(f"[Statistical Summary: .describe(include='all') for '{table_name}']:")
    desc_df = df.describe(include="all").T
    avail_cols = [c for c in ["count", "unique", "top", "freq", "mean", "std", "min", "25%", "50%", "75%", "max"] if c in desc_df.columns]
    desc_display = desc_df[avail_cols].fillna("-")
    display(desc_display)
    
    if show_sample:
        print(f"\n[Head Sample (3 Rows) for '{table_name}']:")
        display(df.head(3))
        
    print("\n" + "-" * 95 + "\n")

# Profile all 14 tables in alphabetical order
print("[EXECUTING] Generating structural profiles for all 14 tables...")
for t_name in sorted(raw_dataframes.keys()):
    profile_f1_table(t_name, raw_dataframes[t_name], show_sample=False)

print("[SUCCESS] Structural profiling completed across all 14 tables.")

### 2.1.5 Focal Deep Dive: Structural Audit of Key Pipeline Tables

To prepare for downstream cleaning and feature engineering, we inspect the three pivotal tables identified in the project roadmap:

| Target Table | Analytical Role | Primary Structural Challenge |
| :--- | :--- | :--- |
| **`results`** | Pipeline Spine & Target Generator | 176 shared-drive duplicate `(raceId, driverId)` pairs; 11 post-race blacklisted columns. |
| **`lap_times`** | Historical In-Race Telemetry | 589k records; must be confirmed strictly non-accessible to feature engineering. |
| **`qualifying`** | Primary Starting-Performance Signal | Era truncation (starts 1994); expected structural nulls in knockout rounds `q2` and `q3`. |

In [ ]:
# ==============================================================================
# 2.1.6 TASK 1.4: FOCUSED AUDIT ON CORE PIPELINE TABLES (RESULTS, LAP_TIMES, QUALIFYING)
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. RESULTS TABLE (The Pipeline Spine)
# ------------------------------------------------------------------------------
results_df = raw_dataframes["results"]
print("=" * 90)
print("1. RESULTS TABLE AUDIT (SPINE OF THE PIPELINE)")
print("=" * 90)
print(f"Total Rows       : {len(results_df):,}")
print(f"Total Columns    : {len(results_df.columns)}")
print(f"Unique Races     : {results_df['raceId'].nunique():,}")
print(f"Unique Drivers   : {results_df['driverId'].nunique():,}")

# Check composite grain duplicates in raw state
dup_count = results_df.duplicated(subset=["raceId", "driverId"]).sum()
print(f"Composite Key (raceId, driverId) Duplicates: {dup_count:,} duplicate rows")
print(f"-> Confirms Trap 1 (shared drives in 1950s). Deduplication rule required in Task 6.")

# Check target distribution in raw state
raw_podium_count = (results_df["positionOrder"] <= 3).sum()
print(f"Podium Target Instances (positionOrder <= 3): {raw_podium_count:,} ({raw_podium_count / len(results_df) * 100:.2f}%)")

# Inspect non-starter indicators (positionText)
pos_text_counts = results_df["positionText"].value_counts().head(10).to_dict()
print(f"Top 10 positionText values: {pos_text_counts}")

display(results_df.head(3))

# ------------------------------------------------------------------------------
# 2. LAP_TIMES TABLE (Largest Volume / 100% In-Race Telemetry)
# ------------------------------------------------------------------------------
lap_times_df = raw_dataframes["lap_times"]
print("\n" + "=" * 90)
print("2. LAP_TIMES TABLE AUDIT (IN-RACE TELEMETRY)")
print("=" * 90)
print(f"Total Rows       : {len(lap_times_df):,}")
print(f"Total Columns    : {len(lap_times_df.columns)} -> {list(lap_times_df.columns)}")
print(f"Memory Footprint : {lap_times_df.memory_usage(deep=True).sum() / (1024*1024):.2f} MB")
print(f"Lap Number Range : min={lap_times_df['lap'].min()}, max={lap_times_df['lap'].max()}")
print(f"-> LEAKAGE CONFIRMATION: lap_times contains exclusively in-race telemetry.")
print(f"-> Verified: 0 columns from lap_times are eligible for pre-race modeling.")

display(lap_times_df.head(3))

# ------------------------------------------------------------------------------
# 3. QUALIFYING TABLE (Pre-Race Feature Spine & Era Coverage Drift)
# ------------------------------------------------------------------------------
qualifying_df = raw_dataframes["qualifying"]
races_df = raw_dataframes["races"]
print("\n" + "=" * 90)
print("3. QUALIFYING TABLE AUDIT (COVERAGE & DRIFT)")
print("=" * 90)
print(f"Total Rows       : {len(qualifying_df):,}")
print(f"Columns          : {list(qualifying_df.columns)}")

# Merge with races to inspect era coverage
quali_races = qualifying_df.merge(races_df[["raceId", "year"]], on="raceId", how="left")
min_quali_year = int(quali_races["year"].min())
max_quali_year = int(quali_races["year"].max())
print(f"Qualifying Coverage Span: {min_quali_year} to {max_quali_year}")
print(f"Total Championship Years: {int(races_df['year'].min())} to {int(races_df['year'].max())}")
print(f"-> Era Gap: Years {int(races_df['year'].min())} to {min_quali_year - 1} have 0 qualifying session rows!")
print(f"-> Historical starting position for 1950-1993 must rely on results.grid.")

# Knockout qualifying null breakdown
q_nulls = qualifying_df[["q1", "q2", "q3"]].isna().sum().to_dict()
print(f"Qualifying Knockout Session Nulls:")
for q_col, n_null in q_nulls.items():
    print(f"  {q_col:<4}: {n_null:>5,} missing ({n_null / len(qualifying_df) * 100:.2f}%)")
print(f"-> Structural Missingness: q2/q3 nulls reflect eliminated drivers, not corrupted data.")

display(qualifying_df.head(3))

---

### Task 1 Completion Summary & Transition to Task 2

**Summary of Structural Audit Deliverables:**
- ✅ **Table Inventory Verified:** Programmatically confirmed `raw_dataframes` contains exactly 14 tables matching canonical Ergast schema names.
- ✅ **Structural Summary Matrix:** Compiled matrix of row counts, column counts, memory footprints, missing cell counts, null rates, and data type breakdowns.
- ✅ **Per-Table Profiling Suite:** Standardized `.info()` and `.describe(include='all')` profiles generated across all 14 tables.
- ✅ **Core Tables Deep Dive:** Quantified `results` spine & 176 shared drive duplicates, confirmed `lap_times` in-race telemetry boundary, and documented `qualifying` era coverage drift (1994+) and knockout missingness.

**Next Task in Step 1:**
The pipeline advances to **Task 2: Primary Key & Referential Integrity Checks**:
- Primary key uniqueness and non-null assertions across all 14 tables (`circuitId`, `constructorId`, `driverId`, `raceId`, `resultId`, etc.).
- Composite key uniqueness checks (`(raceId, driverId, lap)` in `lap_times`, `(raceId, driverId, stop)` in `pit_stops`).
- Foreign key anti-join reconciliation across all parent-child relationships, asserting zero orphan records.